In [2]:
import os, glob
import cv2
import numpy as np

IMG_DIR = r"C:\Users\DELL\derme\tests\demo_images"  

paths = sorted(p for ext in ("*.jpg", "*.jpeg", "*.png")
               for p in glob.glob(os.path.join(IMG_DIR, ext)))
names = [os.path.basename(p) for p in paths]
images = [cv2.imread(p) for p in paths]
assert images, "No images found - check IMG_DIR"
print(f"Loaded {len(images)} images:", names)

chosen = {}   


def nothing(value):
    pass


def blur_score(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    scale = 512 / max(h, w)
    if scale < 1:
        gray = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compose(img, lines, ok=True, min_w=620, max_h=800):
    h, w = img.shape[:2]
    if h > max_h:
        s = max_h / h
        img = cv2.resize(img, (int(w * s), max_h), interpolation=cv2.INTER_AREA)
        h, w = img.shape[:2]
    canvas = np.zeros((h + 90, max(w, min_w), 3), np.uint8)
    canvas[90:, :w] = img
    color = (0, 200, 0) if ok else (0, 0, 255)
    for i, line in enumerate(lines):
        cv2.putText(canvas, line, (8, 26 + i * 26), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color if i == 0 else (255, 255, 255), 2 if i == 0 else 1)
    return canvas


def close_window():
    cv2.destroyAllWindows()
    for i in range(4):      
        cv2.waitKey(1)

Loaded 11 images: ['01_test.jpg', '02_test.jpg', '03_test.jpg', '04_test .jpg', '05_test .jpg', '06_test .jpg', '07_test.jpg', '08_test .jpg', '09_test .jpg', '10_test .jpg', '11_test .jpg']


In [5]:
WINDOW = "1 - Blur   (q = quit)"

cv2.namedWindow(WINDOW)
cv2.createTrackbar("image", WINDOW, 0, max(len(images) - 1, 1), nothing)
cv2.createTrackbar("blur", WINDOW, 0, 30, nothing)
cv2.createTrackbar("blur thr", WINDOW, 10, 300, nothing)

threshold = 50
while True:
    i = min(cv2.getTrackbarPos("image", WINDOW), len(images) - 1)
    sigma = cv2.getTrackbarPos("blur", WINDOW)
    threshold = cv2.getTrackbarPos("blur thr", WINDOW)

    img = images[i].copy()
    if sigma > 0:
        k = int(sigma * 6) | 1            # kernel size must be odd
        img = cv2.GaussianBlur(img, (k, k), sigma)

    score = blur_score(img)
    ok = score >= threshold
    cv2.imshow(WINDOW, compose(img, [
        f"blur score {score:.1f}  |  threshold {threshold}  |  {'PASS' if ok else 'FAIL: too blurry'}",
        f"blur amount {sigma}",
        names[i]], ok))

    if cv2.waitKey(30) & 0xFF in (ord("q"), 27):
        break
    if cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
        break

chosen["BLUR_THRESHOLD"] = threshold
close_window()
print("BLUR_THRESHOLD =", threshold)

BLUR_THRESHOLD = 15


In [7]:
WIN = "2 - Brightness   (q = quit)"
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
cv2.createTrackbar("image", WINDOW, 0, max(len(images) - 1, 1), nothing)
cv2.createTrackbar("brightness", WINDOW, 150, 300, nothing)
cv2.createTrackbar("dark thr", WINDOW, 60, 255, nothing)
cv2.createTrackbar("bright thr", WINDOW, 200, 255, nothing)

dark_threshold, bright_threshold = 60, 200
while True:
    i = min(cv2.getTrackbarPos("image", WINDOW), len(images) - 1)
    shift = cv2.getTrackbarPos("brightness", WINDOW) - 150
    dark_threshold = cv2.getTrackbarPos("dark thr", WINDOW)
    bright_threshold = cv2.getTrackbarPos("bright thr", WINDOW)

    img = cv2.convertScaleAbs(images[i], alpha=1.0, beta=shift)
    mean_b = float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())

    if mean_b < dark_threshold:
        status, ok = "FAIL: too dark", False
    elif mean_b > bright_threshold:
        status, ok = "FAIL: too bright", False
    else:
        status, ok = "PASS", True
    cv2.imshow(WINDOW, compose(img, [
        f"mean brightness {mean_b:.0f}  |  dark thr {dark_threshold}  bright thr {bright_threshold}  |  {status}",
        f"brightness shift {shift}",
        names[i]], ok))

    if cv2.waitKey(30) & 0xFF in (ord("q"), 27):
        break
    if cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
        break

chosen["DARK_MEAN"] = dark_threshold
chosen["BRIGHT_MEAN"] = bright_threshold
close_window()
print("DARK_MEAN =", dark_threshold, " BRIGHT_MEAN =", bright_threshold)


DARK_MEAN = 80  BRIGHT_MEAN = 160


In [8]:
WINDOW = "3 - Size   (q = quit)"
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
cv2.createTrackbar("image", WINDOW, 0, max(len(images) - 1, 1), nothing)
cv2.createTrackbar("size %", WINDOW, 100, 150, nothing)
cv2.createTrackbar("min side", WINDOW, 300, 1000, nothing)

min_side = 300
while True:
    i = min(cv2.getTrackbarPos("image", WINDOW), len(images) - 1)
    pct = max(5, cv2.getTrackbarPos("size %", WINDOW))
    min_side = cv2.getTrackbarPos("min side", WINDOW)

    h, w = images[i].shape[:2]
    nw, nh = max(1, int(w * pct / 100)), max(1, int(h * pct / 100))
    interp = cv2.INTER_AREA if pct < 100 else cv2.INTER_LINEAR
    small = cv2.resize(images[i], (nw, nh), interpolation=interp)

    side = min(nw, nh)
    ok = side >= min_side
    cv2.imshow(WINDOW, compose(small[:800, :700], [
        f"{nw}x{nh}px  shortest side {side}  |  min side {min_side}  |  {'PASS' if ok else 'FAIL: too small'}",
        f"size {pct}%  (shown at true pixel size)",
        names[i]], ok, max_h=10**6))

    if cv2.waitKey(30) & 0xFF in (ord("q"), 27):
        break
    if cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
        break

chosen["MIN_SIDE_PX"] = min_side
close_window()
print("MIN_SIDE_PX =", min_side)

MIN_SIDE_PX = 451


In [3]:
WINDOW = "4 - Skin   (q = quit)"
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
cv2.createTrackbar("image", WINDOW, 0, max(len(images) - 1, 1), nothing)
cv2.createTrackbar("cr low", WINDOW, 133, 160, nothing)
cv2.createTrackbar("cb high", WINDOW, 127, 160, nothing)
cv2.createTrackbar("min skin %", WINDOW, 8, 100, nothing)

min_skin = 8
while True:
    i = min(cv2.getTrackbarPos("image", WINDOW), len(images) - 1)
    cr_low = cv2.getTrackbarPos("cr low", WINDOW)
    cb_high = cv2.getTrackbarPos("cb high", WINDOW)
    min_skin = cv2.getTrackbarPos("min skin %", WINDOW)

    img = images[i]
    h, w = img.shape[:2]
    if h > 600:                                   
        img = cv2.resize(img, (int(w * 600 / h), 600), interpolation=cv2.INTER_AREA)

    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    mask = cv2.inRange(ycrcb, (0, cr_low, 77), (255, 173, cb_high))
    skin_pct = 100.0 * mask.mean() / 255.0

    overlay = img.copy()
    overlay[mask > 0] = (0, 255, 0)
    ok = skin_pct >= min_skin
    cv2.imshow(WINDOW, compose(np.hstack([img, overlay]), [
        f"detected skin {skin_pct:.0f}%  |  min skin {min_skin}%  |  {'PASS' if ok else 'FAIL: not enough skin'}",
        f"cr low {cr_low}   cb high {cb_high}",
        names[i]], ok, min_w=700))

    if cv2.waitKey(30) & 0xFF in (ord("q"), 27):
        break
    if cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE) < 1:
        break

chosen["MIN_SKIN_FRACTION"] = round(min_skin / 100, 2)
close_window()
print("MIN_SKIN_FRACTION =", chosen["MIN_SKIN_FRACTION"])

MIN_SKIN_FRACTION = 0.08


In [14]:
print("Copy these into config.py:\n")
for key in ["BLUR_THRESHOLD", "DARK_MEAN", "BRIGHT_MEAN", "MIN_SIDE_PX", "MIN_SKIN_FRACTION"]:
    print(f"{key} = {chosen.get(key, 'not set yet - run its cell')}")


Copy these into config.py:

BLUR_THRESHOLD = 15
DARK_MEAN = 80
BRIGHT_MEAN = 160
MIN_SIDE_PX = 451
MIN_SKIN_FRACTION = 0.2
